# Mixture of Experts: More Parameters, Same FLOPs

The full code of [Mixture of Experts: More Parameters, Same FLOPs](https://residual-stream.lvtan71.workers.dev/posts/mixture-of-experts/), run from [its folder](https://github.com/lvtan71/residual-stream-code/tree/main/posts/mixture-of-experts) in the
order the folder's README gives. Each script is followed by the output it
printed for the post, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/mixture-of-experts.ipynb)

**Hardware:** Use a GPU runtime (Colab: Runtime → Change runtime type → T4 GPU). About 240 min end to end on the RTX 3090 machine that saved the outputs below.
balancing.py and wikitext2_runs.py train on the GPU; overfit.py trains on the CPU, about an hour on the post's laptop and longer on most cloud machines.

The post's numbers come from an Apple M4 laptop. Other hardware and library
versions change timings and the last digits of some floating-point results.
Sampled text and small training runs can move further; where that changes a
number the post discusses, the note above says so.

**Outputs saved from a run on NVIDIA GeForce RTX 3090 on 2026-10-18** (torch 2.14.1+cu130, 95 min). Every cell ran except `overfit.py`, which trains on the CPU for about an hour on an Apple M4 and was stopped to save rental time; the post's output is below its cell. Run the notebook to replace them with your own.

In [1]:
%pip install -q "torch==2.14.*" "tokenizers==0.23.*" huggingface_hub pyarrow

# Setup: fetch the companion repo and work in this post's folder.
import os, subprocess, sys
SLUG = "mixture-of-experts"
ROOT = os.path.abspath("..") if os.path.isdir("../posts") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
os.chdir(os.path.join(ROOT, "posts", SLUG))
print("working in", os.getcwd())

Note: you may need to restart the kernel to use updated packages.
working in residual-stream-code/posts/mixture-of-experts


## `balancing.py`

The tiny GPT with 8 experts in place of each MLP, on the blog's characters: top-1 with the balancing loss, with no balancing and with bias balancing, and top-2 with the balancing loss, two seeds each, 4,000 steps on the dev slice. Prints bits per byte at the checkpoints and the busiest expert's share in each layer.

In [2]:
!{sys.executable} balancing.py

8 experts, top-1, balancing loss   s0: 1000: 2.102 / 2000: 2.101 / 3000: 2.294 / 4000: 2.532 / busiest expert: 16% 14% 14% 13%  (61 passes)


8 experts, top-1, no balancing     s0: 1000: 2.141 / 2000: 2.045 / 3000: 2.076 / 4000: 2.095 / busiest expert: 64% 98% 98% 62%  (61 passes)


8 experts, top-1, bias balancing   s0: 1000: 2.106 / 2000: 2.101 / 3000: 2.313 / 4000: 2.587 / busiest expert: 14% 14% 15% 14%  (61 passes)


8 experts, top-2, balancing loss   s0: 1000: 2.048 / 2000: 2.097 / 3000: 2.317 / 4000: 2.541 / busiest expert: 14% 16% 15% 14%  (61 passes)


8 experts, top-1, balancing loss   s1: 1000: 2.103 / 2000: 2.109 / 3000: 2.304 / 4000: 2.536 / busiest expert: 15% 14% 16% 14%  (61 passes)


8 experts, top-1, no balancing     s1: 1000: 2.129 / 2000: 2.043 / 3000: 2.113 / 4000: 2.135 / busiest expert: 99% 87% 100% 40%  (61 passes)


8 experts, top-1, bias balancing   s1: 1000: 2.126 / 2000: 2.108 / 3000: 2.304 / 4000: 2.537 / busiest expert: 14% 15% 13% 14%  (61 passes)


8 experts, top-2, balancing loss   s1: 1000: 2.058 / 2000: 2.121 / 3000: 2.323 / 4000: 2.588 / busiest expert: 15% 16% 15% 16%  (61 passes)


The post printed:

```text
8 experts, top-1, balancing loss   s0: 1000: 2.091 / 2000: 2.100 / 3000: 2.279 / 4000: 2.527 / busiest expert: 17% 15% 14% 15%  (61 passes)
8 experts, top-1, no balancing     s0: 1000: 2.146 / 2000: 2.059 / 3000: 2.059 / 4000: 2.102 / busiest expert: 61% 98% 99% 75%  (61 passes)
8 experts, top-1, bias balancing   s0: 1000: 2.110 / 2000: 2.120 / 3000: 2.311 / 4000: 2.561 / busiest expert: 15% 14% 15% 14%  (61 passes)
8 experts, top-2, balancing loss   s0: 1000: 2.050 / 2000: 2.116 / 3000: 2.315 / 4000: 2.567 / busiest expert: 15% 14% 15% 14%  (61 passes)
8 experts, top-1, balancing loss   s1: 1000: 2.106 / 2000: 2.109 / 3000: 2.303 / 4000: 2.549 / busiest expert: 14% 16% 16% 15%  (61 passes)
8 experts, top-1, no balancing     s1: 1000: 2.143 / 2000: 2.055 / 3000: 2.095 / 4000: 2.132 / busiest expert: 100% 86% 100% 41%  (61 passes)
8 experts, top-1, bias balancing   s1: 1000: 2.129 / 2000: 2.137 / 3000: 2.339 / 4000: 2.541 / busiest expert: 15% 15% 15% 14%  (61 passes)
8 experts, top-2, balancing loss   s1: 1000: 2.077 / 2000: 2.124 / 3000: 2.317 / 4000: 2.548 / busiest expert: 15% 15% 16% 15%  (61 passes)
```

## `overfit.py`

The dense model and the top-1 MoE again, seed 0, on the CPU, also scoring 60,000 characters of the training text at each checkpoint.

In [3]:
!{sys.executable} overfit.py

The post printed:

```text
dense                              s0: 1000: 2.145 / 1000 on training text: 1.807 / 2000: 2.047 / 2000 on training text: 1.576 / 3000: 2.051 / 3000 on training text: 1.455 / 4000: 2.029 / 4000 on training text: 1.365  (61 passes)
8 experts, top-1, balancing loss   s0: 1000: 2.095 / 1000 on training text: 1.658 / 2000: 2.104 / 2000 on training text: 1.228 / 3000: 2.292 / 3000 on training text: 0.915 / 4000: 2.553 / 4000 on training text: 0.642 / busiest expert: 17% 15% 14% 15%  (61 passes)
```

## `wikitext2.py`

Downloads WikiText-2 (raw) at a pinned revision and writes `wikitext2_train.txt` and `wikitext2_test.txt`.

In [4]:
!{sys.executable} wikitext2.py

10892990 1285622 passes in 4000 steps of 32x256: 3.3


The post printed:

```text
10892990 1285622 passes in 4000 steps of 32x256: 3.3
```

## `wikitext2_runs.py`

The dense model and the MoE, top-1 and top-2, on WikiText-2, two seeds each, 4,000 steps. Reads the files `wikitext2.py` writes.

In [5]:
!{sys.executable} wikitext2_runs.py

dense (WikiText-2)                 s0: 1000: 2.216 / 2000: 2.066 / 3000: 2.011 / 4000: 1.973  (3 passes)


8 experts, top-1, balancing loss   s0: 1000: 2.157 / 2000: 1.987 / 3000: 1.927 / 4000: 1.895 / busiest expert: 16% 14% 14% 14%  (3 passes)


8 experts, top-2, balancing loss   s0: 1000: 2.085 / 2000: 1.946 / 3000: 1.891 / 4000: 1.853 / busiest expert: 15% 14% 13% 15%  (3 passes)


dense (WikiText-2)                 s1: 1000: 2.219 / 2000: 2.096 / 3000: 2.008 / 4000: 1.982  (3 passes)


8 experts, top-1, balancing loss   s1: 1000: 2.148 / 2000: 1.998 / 3000: 1.928 / 4000: 1.892 / busiest expert: 15% 14% 16% 14%  (3 passes)


8 experts, top-2, balancing loss   s1: 1000: 2.099 / 2000: 1.970 / 3000: 1.896 / 4000: 1.858 / busiest expert: 14% 14% 14% 14%  (3 passes)


The post printed:

```text
dense (WikiText-2)                 s0: 1000: 2.202 / 2000: 2.063 / 3000: 2.006 / 4000: 1.974  (3 passes)
8 experts, top-1, balancing loss   s0: 1000: 2.146 / 2000: 1.990 / 3000: 1.922 / 4000: 1.885 / busiest expert: 20% 18% 16% 14%  (3 passes)
8 experts, top-2, balancing loss   s0: 1000: 2.083 / 2000: 1.943 / 3000: 1.905 / 4000: 1.860 / busiest expert: 15% 14% 13% 15%  (3 passes)
dense (WikiText-2)                 s1: 1000: 2.217 / 2000: 2.088 / 3000: 2.009 / 4000: 1.979  (3 passes)
8 experts, top-1, balancing loss   s1: 1000: 2.154 / 2000: 2.015 / 3000: 1.937 / 4000: 1.898 / busiest expert: 19% 15% 15% 14%  (3 passes)
8 experts, top-2, balancing loss   s1: 1000: 2.095 / 2000: 1.966 / 3000: 1.900 / 4000: 1.862 / busiest expert: 14% 13% 13% 14%  (3 passes)
```

## `moe_arithmetic.py`

Mixtral 8x7B's and DeepSeek-V3's expert weights, total and used per token; the all-to-all traffic per token under expert parallelism; and how many experts a decode batch touches.

In [6]:
!{sys.executable} moe_arithmetic.py

Mixtral 8x7B: one expert 176M weights; all experts, all layers 45.1B; used per token 11.3B
DeepSeek-V3: one routed expert 44M weights; routed experts, all layers 654B of the reported 671B; MLP weights used per token 24.2B of the reported 37B active
all-to-all per token: 168 KiB per MoE layer, 9.5 MiB across the 58 MoE layers (at most; tokens whose experts share a GPU send less)
batch     1: about   8.0 of 256 experts touched per layer,    1.0 tokens per touched expert
batch     8: about  57.4 of 256 experts touched per layer,    1.1 tokens per touched expert
batch    64: about 222.4 of 256 experts touched per layer,    2.3 tokens per touched expert
batch   512: about 256.0 of 256 experts touched per layer,   16.0 tokens per touched expert
batch 4,096: about 256.0 of 256 experts touched per layer,  128.0 tokens per touched expert


The post printed:

```text
Mixtral 8x7B: one expert 176M weights; all experts, all layers 45.1B; used per token 11.3B
DeepSeek-V3: one routed expert 44M weights; routed experts, all layers 654B of the reported 671B; MLP weights used per token 24.2B of the reported 37B active
all-to-all per token: 168 KiB per MoE layer, 9.5 MiB across the 58 MoE layers (at most; tokens whose experts share a GPU send less)
batch     1: about   8.0 of 256 experts touched per layer,    1.0 tokens per touched expert
batch     8: about  57.4 of 256 experts touched per layer,    1.1 tokens per touched expert
batch    64: about 222.4 of 256 experts touched per layer,    2.3 tokens per touched expert
batch   512: about 256.0 of 256 experts touched per layer,   16.0 tokens per touched expert
batch 4,096: about 256.0 of 256 experts touched per layer,  128.0 tokens per touched expert
```